# Step 2.1 Extension — Rolling Data Preparation

This notebook adapts the baseline data preparation to the coursework extension: instead of one train/test split using 20 stocks, we process the full available universe and build rolling monthly train/test windows.

The scaling variables follow the exercise methodology. For each stock-day, we compute:

$$
\text{px\_vol}_{i,d} = \operatorname{std}_t(r_{i,d,t})
$$

where $r_{i,d,t}$ are intraday 10-second returns, and

$$
\text{volume}_{i,d} = \sum_t |q_{i,d,t}|.
$$

Then we compute backward-looking rolling averages over the previous 20 trading days and shift by one day to avoid look-ahead bias. These become the daily scaling factors used later in impact fitting:

$$
\sigma_{i,d} = \frac{1}{20} \sum_{k=1}^{20} \text{px\_vol}_{i,d-k},
\qquad
ADV_{i,d} = \frac{1}{20} \sum_{k=1}^{20} \text{volume}_{i,d-k}.
$$

Monthly trade and price panels are still saved separately to make later rolling fitting and backtesting efficient.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

import importlib

import src.data_prep_rolling_extension as data_prep_rolling_extension

importlib.reload(data_prep_rolling_extension)

from src.data_prep_rolling_extension import *

## 1. Set paths

Set `bin_sample_path` to the folder containing files named like `bin201901.csv`, `bin201902.csv`, etc. The outputs are saved in `rolling_result_path`.

In [8]:
bin_sample_path = "data/binSamples"
rolling_result_path = "data/rolling_data_preparation"

os.makedirs(rolling_result_path, exist_ok=True)

## 2. Find available months

In [7]:
months_df = find_available_bin_months(bin_sample_path)
display(months_df)

,year,month,year_month,file_name
0,2019,1,201901,bin201901.csv
1,2019,2,201902,bin201902.csv
2,2019,3,201903,bin201903.csv
3,2019,4,201904,bin201904.csv
4,2019,5,201905,bin201905.csv
5,2019,6,201906,bin201906.csv
6,2019,7,201907,bin201907.csv
7,2019,8,201908,bin201908.csv
8,2019,9,201909,bin201909.csv
9,2019,10,201910,bin201910.csv


## 3. Process all months over the full universe

For each month, this creates:

- `trade_panel_YYYYMM.csv`
- `price_panel_YYYYMM.csv`
- `daily_stock_info_YYYYMM.csv`

The daily stock-info file contains the exercise-style quantities `px_vol` and `volume`.

In [9]:
months_df, monthly_diagnostics_df = process_all_full_universe_months(
    bin_sample_path=bin_sample_path,
    output_path=rolling_result_path,
)

display(monthly_diagnostics_df)

Processing 201901...
Saved 201901: 50 stocks, 1050 stock-days, 2341 time bins
Processing 201902...
Saved 201902: 50 stocks, 950 stock-days, 2341 time bins
Processing 201903...
Saved 201903: 50 stocks, 1050 stock-days, 2341 time bins
Processing 201904...
Saved 201904: 50 stocks, 1050 stock-days, 2341 time bins
Processing 201905...
Saved 201905: 50 stocks, 1100 stock-days, 2341 time bins
Processing 201906...
Saved 201906: 50 stocks, 1000 stock-days, 2341 time bins
Processing 201907...
Saved 201907: 50 stocks, 1100 stock-days, 2341 time bins
Processing 201908...
Saved 201908: 50 stocks, 1084 stock-days, 2341 time bins
Processing 201909...
Saved 201909: 49 stocks, 980 stock-days, 2341 time bins
Processing 201910...
Saved 201910: 49 stocks, 1127 stock-days, 2341 time bins
Processing 201911...
Saved 201911: 49 stocks, 980 stock-days, 2341 time bins
Processing 201912...
Saved 201912: 49 stocks, 931 stock-days, 2341 time bins


,year_month,n_rows_raw,n_stock_days,n_time_bins,n_stocks,total_abs_volume,trade_file,price_file,daily_stock_info_file
0,201901,2168002,1050,2341,50,538759147.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
1,201902,2018914,950,2341,50,415172779.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
2,201903,2258058,1050,2341,50,465897772.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
3,201904,2185122,1050,2341,50,408962599.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
4,201905,2364583,1100,2341,50,524344513.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
5,201906,2103978,1000,2341,50,436886509.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
6,201907,2247986,1100,2341,50,424206393.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
7,201908,2343350,1084,2341,50,491355380.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
8,201909,2021472,980,2341,49,366080999.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...
9,201910,2321777,1127,2341,49,388501095.0,data/rolling_data_preparation\trade_panel_2019...,data/rolling_data_preparation\price_panel_2019...,data/rolling_data_preparation\daily_stock_info...


## 4. Compute rolling scaling

This builds one file with daily no-look-ahead scaling factors:

- `rolling_scaling_20d.csv`

The first 20 days for each stock will not have scaling values because the rolling window needs 20 previous observations.

In [ ]:
num_days_precompute = 20

rolling_scaling_df = save_rolling_scaling(
    output_path=rolling_result_path,
    months_df=months_df,
    num_days_precompute=num_days_precompute,
)

display(rolling_scaling_df.head())
print(rolling_scaling_df.shape)

## 5. Build the rolling train/test manifest

Each row uses month `m` as the training month and the next available month `m+1` as the testing month.

In [ ]:
rolling_manifest_df = build_rolling_manifest(months_df)

rolling_manifest_df.to_csv(
    os.path.join(rolling_result_path, "rolling_train_test_manifest.csv"),
    index=False,
)

display(rolling_manifest_df)

## 6. Load one rolling window as a test

This returns train and test trade/price panels, plus the daily exercise-style scaling table for the training dates.

In [ ]:
(
    train_trade_df,
    test_trade_df,
    train_px_df,
    test_px_df,
    scaling_df,
    metadata,
) = load_rolling_window(
    output_path=rolling_result_path,
    manifest_df=rolling_manifest_df,
    window_id=0,
    num_days_precompute=num_days_precompute,
)

metadata

In [ ]:
print("Train trade shape:", train_trade_df.shape)
print("Test trade shape :", test_trade_df.shape)
print("Train price shape:", train_px_df.shape)
print("Test price shape :", test_px_df.shape)
print("Scaling shape    :", scaling_df.shape)

print("Missing train trades:", train_trade_df.isna().sum().sum())
print("Missing test trades :", test_trade_df.isna().sum().sum())
print("Missing train prices:", train_px_df.isna().sum().sum())
print("Missing test prices :", test_px_df.isna().sum().sum())

display(scaling_df.head())

## 7. Diagnostics plots

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(monthly_diagnostics_df["year_month"], monthly_diagnostics_df["n_stocks"], marker="o")
plt.xticks(rotation=45)
plt.title("Number of available stocks by month")
plt.xlabel("Month")
plt.ylabel("Number of stocks")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(monthly_diagnostics_df["year_month"], monthly_diagnostics_df["n_stock_days"], marker="o")
plt.xticks(rotation=45)
plt.title("Number of stock-days by month")
plt.xlabel("Month")
plt.ylabel("Stock-days")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(monthly_diagnostics_df["year_month"], monthly_diagnostics_df["total_abs_volume"], marker="o")
plt.xticks(rotation=45)
plt.title("Total absolute traded volume by month")
plt.xlabel("Month")
plt.ylabel("Total absolute traded volume")
plt.tight_layout()
plt.show()

In [ ]:
# Scaling diagnostics
plt.figure(figsize=(8, 4))
plt.hist(rolling_scaling_df["sigma"].dropna(), bins=50)
plt.title("Distribution of exercise-style rolling sigma")
plt.xlabel("sigma")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(rolling_scaling_df["ADV"].dropna(), bins=50)
plt.title("Distribution of exercise-style rolling ADV")
plt.xlabel("ADV")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()